# ЛР № 4. Прогноз структуры причинной модели линейной регрессией

**Что делаем.** В ЛР № 1 мы строили граф, в ЛР № 2 вмешивались в данные и меняли граф по критерию, в ЛР № 3 порождали граф генетическим алгоритмом (обратная задача выбора модели). Здесь решаем **прямую задачу**: обучаем модель на коллекции пар *(выборка, целевая модель)* и по новой выборке **прогнозируем структуру графа** — без перебора и оптимизации по данным.

**Метод (вариант задачи): линейные регрессионные модели.** Прогнозная модель `fai: (выборка, текущий граф) → Γ` — линейная регрессия, где Γ[i, j] — вероятность того, что моном j входит в уравнение i. Прогноз — последовательность графов от полного графа к результату.

Ноутбук запускается быстро: тяжёлая часть (настройка гиперпараметров, проверки на неизвестных системах) посчитана скриптом `run_lab4.py` и лежит в `results/`. Чтобы пересчитать всё заново: `python run_lab4.py --out results --n 30 --seed 0`.

In [ ]:
import sys, json, os
sys.path.insert(0, '.')
import numpy as np
import matplotlib.pyplot as plt
from scmforecast import SYSTEMS, build_collection, hudson_bay, LinearGraphForecaster, Prepared
from scmforecast import viz
from scmforecast.library import make_library, term_name
from scmforecast.experiment import param_error

RES = 'results'
def show(name, width=14):
    img = plt.imread(os.path.join(RES, name))
    h, w = img.shape[:2]
    plt.figure(figsize=(width, width * h / w)); plt.imshow(img); plt.axis('off'); plt.show()

## 1. Коллекция «выборка → целевая модель»

В коллекции 11 систем: Лотка–Вольтерра, её логистический вариант, три варианта с переменной z из ЛР № 2 (z внутри скобок, множитель, аддитивный источник), Ван-дер-Поль, «взаимная любовь» (dx = y, dy = −x), затухающий осциллятор, остывание, Лоренц и Рёсслер. Системы с z, Лоренц и Рёсслер — «модели с большим числом переменных» из задания. У каждой системы каждый экземпляр — свои параметры, начальные условия, шум и число точек. К этому добавлены реальные данные Hudson Bay (21 точка).

Для каждой выборки известна **целевая модель — ground truth**: бинарная матрица Z (какие мономы входят в какое уравнение). Она строится из тех же слагаемых, из которых генерируются данные, поэтому не может разойтись с ними.

In [ ]:
coll = build_collection(1, seed=3)
viz.plot_collection(coll)
plt.show()
print(f"{'система':16s} {'семейство':11s} переменных  рёбер в эталоне  размер библиотеки")
for s in coll:
    print(f"{s.system:16s} {s.family:11s} {s.n_vars:5d} {int(s.Z.sum()):14d} {len(s.terms):14d}")

## 2. Модель и метод

**Модель** линейна по коэффициентам: dx_i/dt = Σ_j w_ij · θ_j(x), где θ_j — мономы степени ≤ 3 от наблюдаемых переменных (для двух переменных 10 мономов, для трёх 20). **Структура** — матрица смежности Z двудольного графа «мономы → уравнения». Граф суперпозиции (как в ЛР № 3) строится из Z: синие вершины — переменные, оранжевые — параметры, зелёные — операции, малиновая обводка — корни dx/dt.

**Параметры** подбираются без производных — интегральным согласованием из ЛР № 3: приращение x за интервал равно интегралу правой части по PCHIP-интерполяции данных. Получается обычная линейная задача.

**Прогнозная модель** `Γ = clip(Φ·β)`: гребневая линейная регрессия по признакам ребра (веса β свои для «ребро сейчас есть» и «ребра сейчас нет»). Признаки не зависят от названий переменных, поэтому одна модель работает для всей коллекции:

| признак | откуда |
|---|---|
| рост ошибки при удалении/добавлении ребра | критерий удаления из ЛР № 3 |
| парный t по квадратам невязок на интервалах | критерий удаления из ЛР № 3 |
| вклад монома, t-статистика коэффициента | линейная модель при текущем графе |
| устойчивость коэффициента по мини-батчам | ЛР № 1–2 |
| вклад в плотной модели, размер уравнения, качество подгонки | |
| степень монома, число переменных, «своя» переменная, драйвер, константа | структура монома |

**Прогноз** — цепочка графов z_T → … → z_0: старт с полного графа; на каждом шаге включаются рёбра с Γ > τ, меняется не больше k самых уверенных; параметры пересчитываются с наследованием (у оставшихся рёбер значения сохраняются, у вылетевших исчезают); в конце — прунинг по критерию из ЛР № 3.

**Обучение** идёт только на совпадении графа с эталоном (ошибка по данным в обучение не входит): состояния z_t получаются прямым процессом (к эталону добавляются ложные рёбра), затем модель дообучается на графах, которые посещает сама, и на копиях выборок с интервенциями в данные.

**Целевая функция J** (по образцу ЛР № 3) настраивает гиперпараметры: `J = ln(acc) + 0.03·C + S`. Здесь acc — доля неверных рёбер, C — сложность спрогнозированных моделей, S = ln(1 + Δ/acc) — устойчивость: на сколько растёт ошибка графа на 8 испорченных копиях выборки (шум, условный шум, малая выборка, смещение × 2 розыгрыша).

## 3. Настройка по J

In [ ]:
tuning = json.load(open(f'{RES}/tuning.json'))
print('Перебрано конфигураций:', len(tuning), '(ridge × τ × k × прунинг); выбор по J на валидации\n')
print(f"{'ridge':>6} {'tau':>4} {'k':>2} {'прунинг':>8} {'J':>7} {'ln acc':>7} {'C':>6} {'S':>6} {'точных':>7} {'F1':>5}")
for r in tuning[:6]:
    print(f"{r['ridge']:6} {r['tau']:4} {r['max_flips']:2d} {str(r['final_prune']):>8} {r['J']:7.3f} {r['ln_acc']:7.3f} {r['C']:6.1f} {r['S']:6.3f} {r['exact']:7.2f} {r['f1']:5.2f}")
show('fig_tuning.png', 12)

J на валидации слабо зависит от ridge (среднее по сетке 0.18 / 0.19 / 0.21 для 0.1 / 1 / 10) и заметно — от порога τ (0.51 / 0.19 / −0.12 для τ = 0.4 / 0.5 / 0.6) и от финального прунинга по критерию ЛР № 3 (в среднем по сетке J = −0.19 с ним и 0.57 без него). Выбранные гиперпараметры сохранены в `results/forecaster.json`.

## 4. Прогноз на новых данных: последовательность графов

In [ ]:
model = LinearGraphForecaster.load(f'{RES}/forecaster.json')
fresh = build_collection(1, seed=2026)          # новые экземпляры, которых не было при обучении
by = {s.system: s for s in fresh}

fc = model.forecast(Prepared(by['van_der_pol']))
print('\n'.join(fc.formula()))
viz.plot_sequence(fc); plt.show()

Сверху — граф модели на каждом шаге (на плотных шагах он слишком велик для рисунка), снизу — матрица смежности: серые клетки — включённые рёбра, красная рамка — эталон, оранжевое — рёбра, удалённые на этом шаге. Видно, как из полного графа убираются лишние мономы, а истинные остаются.

In [ ]:
viz.plot_sequence_metrics(fc); plt.show()
viz.plot_gamma(fc); plt.show()

Слева направо: число рёбер падает к числу рёбер эталона, F1 растёт, необъяснённая доля меняется мало (лишние рёбра почти не уменьшают ошибку — то же наблюдение, что в ЛР № 3). Ниже — вероятности Γ, по которым делались первые шаги.

## 5. Выборка и её аппроксимация спрогнозированной моделью

In [ ]:
fc_l = model.forecast(Prepared(by['lotka']))
viz.plot_fit(fc_l); plt.show()
print('ошибка коэффициентов у угаданных рёбер (до уточнения по траектории): ', param_error(fc_l))

In [ ]:
fc_z = model.forecast(Prepared(by['lotka_z_mult']))
viz.plot_fit(fc_z); plt.show()

## 6. Реальные данные Hudson Bay

In [ ]:
hb = json.load(open(f'{RES}/hudson_bay.json'))
for k, v in hb.items():
    print(k, '->', v.get('formula', ''), '| структура совпала с ЛВ:', v['exact'])
show('fig_hudson_sequence_full.png', 14)
show('fig_hudson_fit_full.png', 12)

Если модель обучена на всей коллекции, она за 5 шагов восстанавливает структуру Лотки–Вольтерры на реальных данных. Коэффициенты после интегрального согласования те же, что в ЛР № 3 (0.61·x − 0.028·xy, −0.95·y + 0.026·xy), а после уточнения по траектории совпадают с эталоном ЛР № 1: RSS ≈ 600 против 595. Все три базовых метода структуру не находят.

**Но:** если из обучения убрать всё семейство Лотки–Вольтерры (включая синтетические экземпляры), модель ошибается: в первое уравнение попадают лишние члены (константа и y). Значит, успех на Hudson Bay — результат того, что система в коллекции есть, а не универсальной способности угадывать неизвестные уравнения.

In [ ]:
show('fig_hudson_sequence_blind.png', 14)

## 7. Сравнение с базовыми методами (тестовая часть коллекции)

In [ ]:
test = json.load(open(f'{RES}/test_metrics.json'))
print(f"{'метод':32s} {'J':>6} {'acc':>5} {'C':>5} {'S':>5} {'точных':>7} {'F1':>5}")
for k, r in test.items():
    print(f"{k:32s} {r['J']:6.2f} {r['acc']:5.2f} {r['C']:5.1f} {r['S']:5.2f} {r['exact']:7.2f} {r['f1']:5.2f}")
show('fig_methods_exact.png', 8)
show('fig_per_system.png', 8)

Линейная модель заметно лучше плотной регрессии с порогом, SINDy и одного критерия ЛР № 3: точных структур ≈ 0.70 против 0.09–0.32, F1 ≈ 0.93 против 0.57–0.67. По системам видно, где метод ошибается: логистическая Лотка–Вольтерра (лишний член x² почти не отличается от шума) и варианты с z при малых амплитудах.

## 8. Устойчивость к интервенциям в данные

In [ ]:
inter = json.load(open(f'{RES}/interventions.json'))
names = list(next(iter(inter.values())))
print(f"{'метод':32s} " + ' '.join(f'{n:>11s}' for n in names))
for m, r in inter.items():
    print(f"{m:32s} " + ' '.join(f"{r[n]['err']:11.2f}" for n in names))
show('fig_interventions.png', 9)

Числа — доля неверных рёбер. Шум, условный шум, малая выборка, смещение и перемешивание портят линейную модель примерно в 3–5 раз (с 0.12 до 0.34–0.65), занулённая вторая переменная ломает её полностью (1.35). Это та же картина, что в ЛР № 1–2: данные с зануленной или перемешанной серией не несут информации о структуре. Базовые методы на чистых данных хуже, чем линейная модель на испорченных (кроме зануления).

## 9. Обобщение на системы, которых не было в обучении

In [ ]:
for f, title in (('leave_family', 'без семейства в обучении'), ('leave_system', 'без системы в обучении')):
    d = json.load(open(f'{RES}/{f}.json'))
    ms = list(next(iter(d.values())))
    print(title, '— доля точных структур')
    print(f"{'':16s} " + ' '.join(f'{m[:14]:>14s}' for m in ms))
    for g, r in d.items():
        print(f"{g:16s} " + ' '.join(f"{r[m]['exact']:14.2f}" for m in ms))
    print(f"{'среднее':16s} " + ' '.join(f"{np.mean([d[g][m]['exact'] for g in d]):14.2f}" for m in ms), '\n')
show('fig_leave_family.png', 9)
show('fig_leave_system.png', 12)

Это самая честная часть проверки. Обучение на коллекции помогает, когда в ней есть **похожие** системы (Лотка–Вольтерра при наличии её вариантов: 0.87; затухающий осциллятор и «любовь»: 1.0; Лоренц: 0.9). Но если неизвестна целая **семья** (без семейства Лотки–Вольтерры 0.18; без единственной системы с одной переменной — остывание — 0.0), модель не лучше, чем критерий ЛР № 3 без обучения: в среднем по семействам 0.40 против 0.41, и на остывании базовые методы лучше. Чтобы прогноз «работал для любых уравнений», коллекция должна быть шире.

## 10. Выводы

1. Линейная регрессионная модель с признаками на основе критерия ЛР № 3 и мини-батчей ЛР № 2 прогнозирует структуру графа по выборке, не перебирая структуры: на тестовой части коллекции угадывает структуру целиком у ≈ 70 % выборок (F1 по рёбрам ≈ 0.93), базовые методы — у 9–32 %.
2. На реальных данных Hudson Bay восстанавливается структура Лотки–Вольтерры; после уточнения параметров RSS ≈ 600 (в ЛР № 1–3: 595).
3. Прогноз — последовательность графов, на каждом шаге параметры наследуются от предыдущего; рёбра, оставшиеся в цепочке, сохраняют значения.
4. Целевая функция J из точности, сложности и устойчивости настраивает гиперпараметры; выбор зависит от весов, и выбор весов остаётся допущением, как в ЛР № 3.
5. Метод хорошо работает на системах, похожих на встречавшиеся в обучении, и слабо — на новых семействах; это ограничение коллекции, а не только метода.
6. Различие структур для малых вмешательств z (ЛР № 2) по данным принципиально неразличимо: в этих случаях точная структура почти не угадывается ни одним методом.